*This code is a companion to the book **Mastering PyTorch and Lightning: A Step-by-Step Practical Guide with QA** by Aghiles Kebaili*
> This notebook contains the raw code for Chapter 2: Tensor Mathematics & Advanced Indexing. To understand how arithmetic, reductions, broadcasting, and indexing behave in real PyTorch code, get the full step-by-step guide on Amazon: **[Get the Book Here](https://www.amazon.fr/Mastering-PyTorch-Lightning-Step-Step-ebook/dp/B0HGNZS55V)**

Tensor operations sit at the core of every PyTorch model. Once you understand element-wise arithmetic, reductions, broadcasting, and slicing, debugging shape mismatches becomes much easier and much less mysterious.

## 1. Tensor Mathematics & Reductions
### Step 1: Separate Element-Wise and Matrix Operations

In [ ]:
import torch

# Element-wise multiplication (*) and matrix multiplication (@) are different operations.
# Mixing them is an easy mistake because both can produce valid-shaped outputs.
# Element-wise combines matching positions; matrix multiplication combines rows and columns.
A = torch.tensor([[1, 2], [3, 4]])
B = torch.tensor([[5, 6], [7, 8]])

# Element-wise multiplication (Hadamard product)
element_wise = A * B
# [[5, 12], [21, 32]]

# Algebraic matrix multiplication
matrix_mult = A @ B  # or torch.matmul(A, B)
# [[19, 22], [43, 50]]

print(element_wise)
print(matrix_mult)

assert torch.equal(
    element_wise,
    torch.tensor([[5, 12], [21, 32]]),
)
assert torch.equal(
    matrix_mult,
    torch.tensor([[19, 22], [43, 50]]),
)

tensor([[ 5, 12],
        [21, 32]])
tensor([[19, 22],
        [43, 50]])


### Step 2: Reduce Dimensions Deliberately

When reading a reduction, identify which dimensions are aggregated and what shape remains. The dim argument selects the dimensions to reduce, while keepdim=True retains them, which is often useful when the result will later be broadcast against the original tensor.

In [ ]:
import torch

# A reduction aggregates values along one or more dimensions.
# dim selects which axes to reduce; keepdim=True preserves those axes with size 1.
# This is useful when the result will later be broadcast against the original tensor.
images = torch.randn(8, 3, 32, 32)

global_mean = images.mean()
gap_features = images.mean(dim=(2, 3))
channel_means = images.mean(dim=(2, 3), keepdim=True)

logits = torch.randn(8, 10)
predictions = torch.argmax(logits, dim=1)

assert global_mean.ndim == 0
assert gap_features.shape == (8, 3)
assert channel_means.shape == (8, 3, 1, 1)
assert predictions.shape == (8,)

print("Global mean:", global_mean.item())
print("GAP features shape:", gap_features.shape)
print("Channel means shape:", channel_means.shape)
print("Predictions shape:", predictions.shape)

Global mean: 0.0033075932879000902
GAP features shape: torch.Size([8, 3])
Channel means shape: torch.Size([8, 3, 1, 1])
Predictions shape: torch.Size([8])


## 2. The Magic of Broadcasting
### Step 1: Predict the Resulting Shape

In [ ]:
import torch

# Broadcasting works by comparing dimensions from right to left.
# Size-1 dimensions can expand to match. It never adds dimensions on the right.
# torch.broadcast_shapes() predicts the result without allocating data.
result_shape = torch.broadcast_shapes((4, 1, 5), (3, 5))

print(result_shape)  # torch.Size([4, 3, 5])
assert result_shape == torch.Size([4, 3, 5])

torch.Size([4, 3, 5])


Broadcasting uses dimension sizes and positions, not labels such as channel or width. Compare shapes from right to left, account for implicit leading dimensions, and insert singleton dimensions when an axis must align with a specific part of a larger tensor.

### Step 2: Align a Channel Vector Explicitly

In [ ]:
import torch

# Broadcasting rules are shape-based only; the names "channel" or "height" have no meaning.
# Alignment failures happen silently and can produce unintended expanded shapes.
# The fix is to explicitly reshape (e.g., view(3, 1, 1)) so dimensions align correctly.
# Image batch: (Batch=8, Channels=3, Height=32, Width=32)
x = torch.randn(8, 3, 32, 32)

# One bias value per channel: (3,)
bias = torch.tensor([0.5, -0.5, 1.0])

try:
    x + bias
except RuntimeError as error:
    print(f"Expected alignment error: {error}")

# (3, 1, 1) is padded on the left to (1, 3, 1, 1).
channel_bias = bias.view(3, 1, 1)
output = x + channel_bias

assert output.shape == x.shape

Expected alignment error: The size of tensor a (32) must match the size of tensor b (3) at non-singleton dimension 3


## 3. Advanced Indexing & Slicing
### Step 1: Select Regular Regions with Slices

In [ ]:
import torch

# Basic slicing describes regular patterns and returns a view backed by the original storage.
# No data is copied; only shape, stride, and storage_offset change.
# Advanced indexing (boolean masks, index tensors) materializes new storage.
batch = torch.randn(64, 3, 32, 32)  # (B, C, H, W)

# First 10 images, every channel and pixel
first_ten = batch[:10, :, :, :]

# Cleaner equivalent with an ellipsis
first_ten_clean = batch[:10, ...]

# Red channel for every image: (64, 32, 32)
red_channel = batch[:, 0, ...]

assert torch.equal(first_ten, first_ten_clean)
assert red_channel.shape == (64, 32, 32)

### Step 2: Filter Irregular Elements with a Boolean Mask

In [ ]:
import torch

# Boolean indexing selects arbitrary elements (not regular patterns) and allocates new storage.
# Modifying the result does not affect the source because they no longer share data.
# This is different from basic slicing, which shares storage.
data = torch.tensor([[1, -2, 3], [-4, 5, -6]])

# Boolean mask with the same shape as data
mask = data > 0
# [[True, False, True], [False, True, False]]

positives = data[mask]
print(positives)  # tensor([1, 3, 5])

positives[0] = 100
assert data[0, 0].item() == 1

tensor([1, 3, 5])


### Step 3: Express Conditional Logic with \texttt{torch.where}

In [ ]:
import torch

# torch.where(condition, x, y) implements element-wise if-else logic.
# It does not remove elements like masking does; it selects one value per position.
# Broadcasting applies to all three arguments (condition, x, y).
x = torch.tensor([[1.0, -2.0], [-3.0, 4.0]])

# Keep x when positive. Otherwise, use 0.0.
relu_emulation = torch.where(x > 0, x, 0.0)

print(relu_emulation)
# tensor([[1., 0.],
#         [0., 4.]])

assert torch.equal(
    relu_emulation,
    torch.tensor([[1.0, 0.0], [0.0, 4.0]]),
)

tensor([[1., 0.],
        [0., 4.]])


### Gotcha 1: The Silent Broadcasting Bug

In [ ]:
import torch

# Silent broadcasting bugs are dangerous: the shapes become valid but the computation is wrong.
# A (64,) and (64,1) tensor become a (64,64) matrix instead of staying as (64,) pairs.
# Always verify shapes when combining tensors in a loss function.
predictions = torch.randn(64)     # (64,)
targets = torch.randn(64, 1)      # (64, 1)

# Broadcasting aligns these as (1, 64) and (64, 1),
# producing an unintended (64, 64) matrix.
residuals = predictions - targets
assert residuals.shape == (64, 64)

# Fix: make the shapes identical before arithmetic.
targets = targets.squeeze(dim=1)  # (64,)
residuals = predictions - targets
loss = residuals.pow(2).mean()
assert residuals.shape == (64,)

### Gotcha 2: Views vs Copies in Indexing

In [ ]:
# Slicing and advanced indexing have opposite aliasing semantics.
# Slicing returns a view that shares storage; advanced indexing allocates new data.
# In-place mutations through a slice affect the source; mutations through indexing do not.
source = torch.arange(6)

basic_slice = source[1:4]
advanced_selection = source[torch.tensor([1, 3])]

basic_slice[0] = 99
assert source[1].item() == 99

advanced_selection[0] = -1
assert source[1].item() == 99